# Historical BE Market Data Fetcher

Fetches Belgium (BE) electricity market data **2022-01-01 → today** via the ENTSO-E
Transparency Platform, cleans it, casts numerics to `float32`, and writes
`data/historical_be_2022_2026.csv.gz`.

## Pipeline

```
ENTSO-E REST API
      │
      ▼  quarterly batches · ~34 sub-queries each
EntsoeDataClient.fetch_comprehensive_market_data()
      │  tenacity retry: exponential backoff 2–10 s · max 3 attempts
      │  + explicit 429 sleep before retry
      ▼
CleaningService.clean_energy_data()
      │  15-min ISP grid · ffill prices/forecasts · NULL-if-NULL physicals
      ▼
check_and_fill_frequency_gaps()   ← reindex + asfreq('15min') + targeted ffill
      │
      ▼
cast_numeric_to_float32()         ← Qlib-ready dtype
      │
      ▼
data/historical_be_2022_2026.csv.gz   (append per batch, crash-safe)
      │
      ▼
Post-process: dedup · sort · asfreq('15min') · dataset_metadata.json
```

## Rate-Limit Strategy

| Layer | Mechanism |
|---|---|
| Sub-query | `tenacity` exponential backoff (2–10 s, ≤3 retries) in `_safe_query` |
| 429 explicit | `time.sleep(RATE_LIMIT_SLEEP_S)` before batch retry |
| Batch | `BATCH_SLEEP_S` sleep after each quarterly batch (default **60 s**) |
| Crash safety | Each batch is **appended immediately**; set `RESUME_FROM_BATCH=N` to restart |

Each quarterly batch ≈ 34 HTTP requests.  
At 60 s sleep → ~34 req/min → well within ENTSO-E limits.  
Raise `BATCH_SLEEP_S` to `120` and `RATE_LIMIT_SLEEP_S` to `300` if you see `429 Too Many Requests`.

## Cell 1 — Imports & Credentials

In [1]:
# Uncomment once to install tqdm if missing:
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm'])

import sys
import os
import io
import json
import time
import logging
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm  # auto: rich bar in Jupyter, plain bar in terminal

# ---------------------------------------------------------------------------
# Project root: notebook lives in <root>/scripts/ — step one level up.
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared_logic.entsoe_client import EntsoeDataClient
from shared_logic.cleaning_service import CleaningService
from shared_logic.constants import DEFAULT_FREQ_GRID

# ---------------------------------------------------------------------------
# Logging — English only, per project Core Principles.
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('historical_fetch')

# ---------------------------------------------------------------------------
# Credential loading — reads ENTSOE_API_KEY from local.settings.json.
# Searches PROJECT_ROOT first, then walks up the directory tree — handles
# git worktrees where local.settings.json lives in the main repo root.
# ---------------------------------------------------------------------------
def load_environment_config() -> None:
    config_path: Path | None = None
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        p = candidate / 'local.settings.json'
        if p.exists():
            config_path = p
            break
    if config_path is not None:
        with open(config_path, 'r') as f:
            settings = json.load(f)
        for k, v in settings.get('Values', {}).items():
            os.environ[k] = str(v)
        logger.info(f'Credentials loaded from {config_path}')
    else:
        logger.warning('local.settings.json not found — relying on shell environment variables.')

load_environment_config()

_api_key: str | None = os.getenv('ENTSOE_API_KEY')
if not _api_key:
    raise EnvironmentError(
        "ENTSOE_API_KEY is not set.\n"
        "  Add it to local.settings.json under 'Values':\n"
        "    { \"Values\": { \"ENTSOE_API_KEY\": \"your_key\" } }\n"
        "  Or export it in your shell before launching Jupyter."
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"API key      : {'*' * 8}{_api_key[-4:]}")
print(f"Python       : {sys.version.split()[0]}")
print(f"pandas       : {pd.__version__}  /  numpy : {np.__version__}")

/Users/bosun/Developer/Quant/energy-data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
21:57:12 [INFO] historical_fetch - Credentials loaded from /Users/bosun/Developer/Quant/energy-data/local.settings.json


Project root : /Users/bosun/Developer/Quant/energy-data
API key      : ********8380
Python       : 3.12.13
pandas       : 3.0.1  /  numpy : 2.4.3


## Cell 2 — Configuration

Set `PREVIEW_ONLY = False` **only after** confirming the Data Health Report in Cell 5.

In [2]:
# ---------------------------------------------------------------------------
# Fetch window
# ---------------------------------------------------------------------------
START_DATE: str = '2022-01-01'
END_DATE:   str = datetime.now(timezone.utc).strftime('%Y-%m-%d')  # today in UTC

# 'QS' = quarterly start (recommended); 'MS' = monthly start (slower, more resilient)
BATCH_FREQ:    str = 'QS'
# Seconds to sleep between batches — raise to 120 on repeated 429 errors.
BATCH_SLEEP_S: int = 60
# Seconds to sleep before retrying a batch that hit a 429 Rate Limit.
RATE_LIMIT_SLEEP_S: int = 180
# Maximum per-batch retries on 429 before giving up and logging as failed.
MAX_BATCH_RETRIES: int = 3

# All timestamps stored as UTC ISO-8601 strings — avoids DST ambiguity.
UTC = 'UTC'

# ---------------------------------------------------------------------------
# Output — gzip saves ~60-70% vs plain CSV; natively supported by Kaggle.
# ---------------------------------------------------------------------------
OUTPUT_PATH: Path = PROJECT_ROOT / 'data' / 'historical_be_2022_2026.csv.gz'
METADATA_PATH: Path = PROJECT_ROOT / 'data' / 'dataset_metadata.json'

# ---------------------------------------------------------------------------
# Reference column list (used for the Data Health Report in Cell 5).
# If the local sample file exists, columns are loaded from it automatically.
# ---------------------------------------------------------------------------
_ref_csv: Path = PROJECT_ROOT / 'data' / 'be_market_data_20260323.csv'
if _ref_csv.exists():
    _ref_df = pd.read_csv(_ref_csv, nrows=0)
    REFERENCE_COLUMNS: list[str] = [c for c in _ref_df.columns if c != 'Time_UTC']
    print(f"Reference columns loaded from {_ref_csv.name}: {len(REFERENCE_COLUMNS)} cols")
else:
    # Derived from entsoe_client._get_query_configs() prefix/column conventions.
    REFERENCE_COLUMNS: list[str] = [
        # Day-ahead market
        'DA_Price',
        # System load
        'Load_Forecast', 'Load_Actual',
        # Net position & imbalance
        'NetPos', 'Imb_imbalance_short', 'Imb_imbalance_long',
        # Wind & solar forecasts (day-ahead)
        'WS_Fc_Solar', 'WS_Fc_Wind_Offshore', 'WS_Fc_Wind_Onshore',
        # Wind & solar intraday forecasts
        'WS_ID_Fc_Solar', 'WS_ID_Fc_Wind_Offshore', 'WS_ID_Fc_Wind_Onshore',
        # Actual generation (key fuel types)
        'Gen_Solar', 'Gen_Wind_Offshore', 'Gen_Wind_Onshore', 'Gen_Nuclear',
        # Aggregated bids & balancing
        'AggBids', 'BalState',
        # Reserve
        'ResPrice', 'ResCap', 'ResAmt',
        # Cross-border flows (physical + scheduled)
        'Export_FR', 'Export_NL', 'Export_DE_LU', 'Export_GB',
        'Import_FR', 'Import_NL', 'Import_DE_LU', 'Import_GB',
        'SchedExc_FR', 'SchedExc_NL', 'SchedExc_DE_LU', 'SchedExc_GB',
        # NTC week-ahead
        'NTC_Week_FR', 'NTC_Week_NL', 'NTC_Week_DE_LU', 'NTC_Week_GB',
        # Physical flows (all-borders aggregate)
        'PhysFlow_All', 'Import_Sum',
    ]
    print(f"Reference CSV not found — using {len(REFERENCE_COLUMNS)} derived columns.")
    print(f"Place be_market_data_20260323.csv in {PROJECT_ROOT / 'data'} to auto-load.")

# ---------------------------------------------------------------------------
# Safety guard: PREVIEW_ONLY=True fetches only 3 days for inspection.
# Set to False only after reviewing the Data Health Report.
# ---------------------------------------------------------------------------
PREVIEW_ONLY: bool = False

print(f"\nFetch window : {START_DATE}  ->  {END_DATE}  (UTC)")
print(f"Batch freq   : {BATCH_FREQ}  |  sleep: {BATCH_SLEEP_S}s  |  429-retry sleep: {RATE_LIMIT_SLEEP_S}s")
print(f"Output       : {OUTPUT_PATH}")
print(f"Metadata     : {METADATA_PATH}")
print(f"Mode         : {'PREVIEW ONLY (3 days)' if PREVIEW_ONLY else 'FULL DOWNLOAD'}")

Reference CSV not found — using 39 derived columns.
Place be_market_data_20260323.csv in /Users/bosun/Developer/Quant/energy-data/data to auto-load.

Fetch window : 2022-01-01  ->  2026-04-09  (UTC)
Batch freq   : QS  |  sleep: 60s  |  429-retry sleep: 180s
Output       : /Users/bosun/Developer/Quant/energy-data/data/historical_be_2022_2026.csv.gz
Metadata     : /Users/bosun/Developer/Quant/energy-data/data/dataset_metadata.json
Mode         : FULL DOWNLOAD


## Cell 3 — Batch Planning

In [3]:
def batch_label(ts: pd.Timestamp, freq: str = BATCH_FREQ) -> str:
    """Return a human-readable label for a batch start timestamp."""
    _qmap = {1: 'Q1', 2: 'Q1', 3: 'Q1', 4: 'Q2', 5: 'Q2', 6: 'Q2',
             7: 'Q3', 8: 'Q3', 9: 'Q3', 10: 'Q4', 11: 'Q4', 12: 'Q4'}
    return f"{ts.year}-{_qmap[ts.month]}" if freq == 'QS' else ts.strftime('%Y-%m')


# Build (start, end) pairs — all boundaries in UTC.
period_starts: pd.DatetimeIndex = pd.date_range(
    start=START_DATE, end=END_DATE, freq=BATCH_FREQ, tz=UTC
)
today_end: pd.Timestamp = pd.Timestamp(END_DATE, tz=UTC) + pd.Timedelta(days=1)

batches: list[tuple[pd.Timestamp, pd.Timestamp]] = [
    (
        period_starts[i],
        period_starts[i + 1] if i + 1 < len(period_starts) else today_end,
    )
    for i in range(len(period_starts))
]

est_minutes: int = len(batches) * (BATCH_SLEEP_S + 90) // 60
print(f"Total batches  : {len(batches)}")
print(f"Date range     : {batches[0][0].date()}  ->  {batches[-1][1].date()}")
print(f"Est. duration  : ~{est_minutes} min  "
      f"({BATCH_SLEEP_S}s sleep + ~90s API latency per batch)")
print()
for i, (s, e) in enumerate(batches, 1):
    print(f"  Batch {i:02d}  {batch_label(s):8s}  {s.date()} -> {e.date()}")

Total batches  : 18
Date range     : 2022-01-01  ->  2026-04-10
Est. duration  : ~45 min  (60s sleep + ~90s API latency per batch)

  Batch 01  2022-Q1   2022-01-01 -> 2022-04-01
  Batch 02  2022-Q2   2022-04-01 -> 2022-07-01
  Batch 03  2022-Q3   2022-07-01 -> 2022-10-01
  Batch 04  2022-Q4   2022-10-01 -> 2023-01-01
  Batch 05  2023-Q1   2023-01-01 -> 2023-04-01
  Batch 06  2023-Q2   2023-04-01 -> 2023-07-01
  Batch 07  2023-Q3   2023-07-01 -> 2023-10-01
  Batch 08  2023-Q4   2023-10-01 -> 2024-01-01
  Batch 09  2024-Q1   2024-01-01 -> 2024-04-01
  Batch 10  2024-Q2   2024-04-01 -> 2024-07-01
  Batch 11  2024-Q3   2024-07-01 -> 2024-10-01
  Batch 12  2024-Q4   2024-10-01 -> 2025-01-01
  Batch 13  2025-Q1   2025-01-01 -> 2025-04-01
  Batch 14  2025-Q2   2025-04-01 -> 2025-07-01
  Batch 15  2025-Q3   2025-07-01 -> 2025-10-01
  Batch 16  2025-Q4   2025-10-01 -> 2026-01-01
  Batch 17  2026-Q1   2026-01-01 -> 2026-04-01
  Batch 18  2026-Q2   2026-04-01 -> 2026-04-10


## Cell 4 — Helper Functions

Pure functions used by both the preview and the full download loop.

In [4]:
def cast_numeric_to_float32(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast all numeric columns to float32 for Qlib compatibility and memory efficiency.
    Non-numeric columns (e.g. Time_Local) are left unchanged.
    """
    numeric_cols: list[str] = df.select_dtypes(include='number').columns.tolist()
    df[numeric_cols] = df[numeric_cols].astype(np.float32)
    return df


def check_and_fill_frequency_gaps(
    df: pd.DataFrame,
    label: str,
    freq: str = '15min',
) -> pd.DataFrame:
    """
    Verify that the DatetimeIndex has no missing 15-min intervals.
    Uses df.asfreq('15min') to ensure a continuous grid, then forward-fills
    step-function columns (prices, forecasts, NTC, reserves) up to 4 periods (1 hour max).
    Physical signals (load actuals, generation, flows) are intentionally left as NaN.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        logger.warning(f'[{label}] Index is not DatetimeIndex — skipping frequency check.')
        return df

    n_before: int = len(df)

    # asfreq introduces NaN rows for any missing 15-min slots.
    df = df.asfreq(freq)
    df.index.name = 'Time_UTC'

    n_inserted: int = len(df) - n_before
    if n_inserted == 0:
        logger.info(f'[{label}] Frequency check passed: continuous {freq} grid confirmed.')
    else:
        logger.warning(
            f'[{label}] {n_inserted} missing {freq} intervals detected and inserted by asfreq.'
        )

    # Forward-fill step-function columns only (market commitments, forecasts).
    # Physical signals must not be filled — business rule: NULL if NULL.
    step_patterns: tuple[str, ...] = (
        'DA_', 'NTC_', 'ResPrice', 'ResCap', 'ResAmt',
        'AggBids', 'SchedExc_', 'Fc_', 'GenFc_', 'Price',
    )
    step_cols: list[str] = [
        c for c in df.columns
        if any(c.startswith(p) or p in c for p in step_patterns)
    ]
    if step_cols:
        df[step_cols] = df[step_cols].ffill(limit=4)  # max 1-hour gap
        logger.info(f'[{label}] Forward-filled {len(step_cols)} step-function columns.')

    return df


def ensure_time_utc_first(df: pd.DataFrame) -> pd.DataFrame:
    """
    Promote the DatetimeIndex to the first column as 'Time_UTC'.
    Returns a DataFrame with Time_UTC as a regular (non-index) column in position 0,
    followed by all metric columns in their existing order.
    This ensures Kaggle consumers can sort/filter on Time_UTC without index gymnastics.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        return df
    df = df.reset_index()
    df = df.rename(columns={'index': 'Time_UTC'})
    if 'Time_UTC' not in df.columns:
        return df
    # Guarantee Time_UTC is at position 0.
    other_cols = [c for c in df.columns if c != 'Time_UTC']
    return df[['Time_UTC'] + other_cols]


def clean_batch(
    raw_df: pd.DataFrame,
    label: str,
) -> pd.DataFrame:
    """
    Full post-fetch processing pipeline for a single batch:
      1. CleaningService (15-min ISP alignment, ffill prices, NULL physicals)
      2. UTC DatetimeIndex normalisation
      3. Frequency gap check using asfreq('15min') + targeted ffill
      4. float32 cast
    Returns a DataFrame indexed by UTC DatetimeIndex.
    """
    # Step 1: run through the production cleaning pipeline.
    cleaned_csv: str = CleaningService.clean_energy_data(raw_df.to_csv(index=True))
    df: pd.DataFrame = pd.read_csv(io.StringIO(cleaned_csv), index_col=0)

    # Step 2: normalise index to UTC DatetimeIndex.
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = 'Time_UTC'
    # Drop Time_Local — keeping only UTC avoids DST complexity downstream.
    df = df.drop(columns=['Time_Local'], errors='ignore')

    # Step 3: check and repair 15-min frequency gaps via asfreq.
    df = check_and_fill_frequency_gaps(df, label)

    # Step 4: cast all numeric columns to float32.
    df = cast_numeric_to_float32(df)

    return df


def is_rate_limit_error(exc: Exception) -> bool:
    """Return True if the exception looks like an HTTP 429 Rate Limit response."""
    msg = str(exc).lower()
    return '429' in msg or 'too many requests' in msg or 'rate limit' in msg


def run_data_health_report(
    df: pd.DataFrame,
    reference_cols: list[str],
    label: str = 'dataset',
) -> dict:
    """
    Print a structured Data Health Report and return a summary dict
    suitable for serialising to dataset_metadata.json.
    """
    sep = '=' * 65
    print(f'\n{sep}')
    print(f'DATA HEALTH REPORT  [{label}]')
    print(sep)

    # Normalise index for reporting regardless of whether it is DatetimeIndex or strings.
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df = df.copy()
            df.index = pd.to_datetime(df.index, utc=True)
        except Exception:
            pass

    metadata: dict = {
        'label': label,
        'generated_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'rows': len(df),
        'columns': len(df.columns),
    }

    # Row / column count
    print(f'  Rows            : {len(df):>8,}')
    print(f'  Columns         : {len(df.columns):>8}')

    # Frequency / time step
    if isinstance(df.index, pd.DatetimeIndex) and len(df) > 1:
        median_step: float = df.index.to_series().diff().dt.total_seconds().median()
        step_min: float = median_step / 60
        step_ok: str = 'OK' if step_min == 15.0 else f'WARNING — expected 15'
        print(f'  Median step     : {step_min:>8.1f} min  [{step_ok}]')
        print(f'  Index TZ        : {str(df.index.tz):>8}  [expected: UTC]')
        print(f'  Index range     : {df.index[0]}  ->  {df.index[-1]}')
        metadata['median_step_min'] = step_min
        metadata['index_tz'] = str(df.index.tz)
        metadata['index_start'] = str(df.index[0])
        metadata['index_end'] = str(df.index[-1])

    # dtype check
    n_float32: int = int((df.dtypes == np.float32).sum())
    n_numeric: int = len(df.select_dtypes(include='number').columns)
    print(f'  float32 cols    : {n_float32:>8} / {n_numeric} numeric')
    metadata['float32_cols'] = n_float32
    metadata['numeric_cols'] = n_numeric

    # Column match vs reference
    matched: list[str] = [c for c in reference_cols if c in df.columns]
    missing: list[str] = [c for c in reference_cols if c not in df.columns]
    extra_n: int = len([c for c in df.columns if c not in reference_cols])

    print(f'\n  Reference column match:')
    print(f'    Matched  : {len(matched):>3} / {len(reference_cols)}')
    if missing:
        print(f'    MISSING  : {missing}')
    else:
        print(f'    MISSING  : none')
    print(f'    Extra    : {extra_n} additional columns not in reference (may be valid)')
    metadata['reference_matched'] = len(matched)
    metadata['reference_total'] = len(reference_cols)
    metadata['reference_missing'] = missing
    metadata['extra_cols'] = extra_n

    # Missing-value summary (top 15 worst)
    missing_pct: pd.Series = df.isnull().mean().mul(100)
    bad: pd.Series = missing_pct[missing_pct > 0].sort_values(ascending=False).head(15)
    print(f'\n  Missing-value summary (top {len(bad)}, >0%):')
    nan_summary: dict = {}
    if bad.empty:
        print('    None — dataset is complete.')
    else:
        for col, pct in bad.items():
            bar = chr(9608) * max(1, int(pct / 5))  # block character
            print(f'    {col:<48s}  {pct:6.1f}%  {bar}')
            nan_summary[col] = round(float(pct), 2)
    metadata['nan_pct_top15'] = nan_summary

    # NaN spike check — flag any column with >50% NaN as a Qlib risk.
    high_nan = {c: v for c, v in nan_summary.items() if v > 50.0}
    if high_nan:
        print(f'\n  *** NaN SPIKE WARNING — {len(high_nan)} columns >50% missing:')
        for c, v in high_nan.items():
            print(f'      {c}: {v:.1f}%  — may break Qlib models')
    metadata['nan_spike_warning'] = high_nan

    print(sep)
    return metadata


print('Helper functions defined:', ', '.join([
    'cast_numeric_to_float32',
    'check_and_fill_frequency_gaps',
    'ensure_time_utc_first',
    'clean_batch',
    'is_rate_limit_error',
    'run_data_health_report',
]))

Helper functions defined: cast_numeric_to_float32, check_and_fill_frequency_gaps, ensure_time_utc_first, clean_batch, is_rate_limit_error, run_data_health_report


## Cell 5 — Preview + Data Health Report

Fetches **3 days** (2022-01-01 to 2022-01-04).  
Review the Data Health Report and `head(10)` before enabling the full download in Cell 2.

In [5]:
PREVIEW_START: pd.Timestamp = pd.Timestamp('2022-01-01', tz=UTC)
PREVIEW_END:   pd.Timestamp = pd.Timestamp('2022-01-04', tz=UTC)  # 3 full days

print(f'Fetching preview: {PREVIEW_START.date()} -> {PREVIEW_END.date()} (3 days) ...')
client: EntsoeDataClient = EntsoeDataClient()
raw_preview: pd.DataFrame = client.fetch_comprehensive_market_data(
    PREVIEW_START, PREVIEW_END
)

if raw_preview is None or raw_preview.empty:
    print('WARNING: Empty API response for the preview window.')
    print('Check ENTSOE_API_KEY validity and ENTSO-E platform availability.')
else:
    df_preview: pd.DataFrame = clean_batch(raw_preview, label='preview')

    # Run the structured Data Health Report.
    run_data_health_report(df_preview, REFERENCE_COLUMNS, label='2022-01-01 to 2022-01-04 (preview)')

    # Show Time_UTC as the first column in the preview.
    df_preview_display = ensure_time_utc_first(df_preview.copy())
    print(f'\nColumn order (first 5): {list(df_preview_display.columns[:5])}')
    print('\n--- head(10) ---')
    try:
        display(df_preview_display.head(10))
    except NameError:
        print(df_preview_display.head(10).to_string())

21:57:31 [INFO] root - Executing comprehensive fetch for BE


Fetching preview: 2022-01-01 -> 2022-01-04 (3 days) ...


21:57:33 [WARNING] shared_logic.entsoe_client - Structural corruption in response: Columns with duplicate values are not supported in stack
21:57:37 [WARNING] shared_logic.entsoe_client - Structural corruption in response: Columns with duplicate values are not supported in stack
21:57:43 [INFO] historical_fetch - [preview] Frequency check passed: continuous 15min grid confirmed.
21:57:43 [INFO] historical_fetch - [preview] Forward-filled 27 step-function columns.



DATA HEALTH REPORT  [2022-01-01 to 2022-01-04 (preview)]
  Rows            :      288
  Columns         :      108
  Median step     :     15.0 min  [OK]
  Index TZ        :      UTC  [expected: UTC]
  Index range     : 2022-01-01 00:00:00+00:00  ->  2022-01-03 23:45:00+00:00
  float32 cols    :      108 / 108 numeric

  Reference column match:
    Matched  :  25 / 39
    MISSING  : ['Load_Forecast', 'Imb_imbalance_short', 'Imb_imbalance_long', 'Gen_Solar', 'Gen_Wind_Offshore', 'Gen_Wind_Onshore', 'Gen_Nuclear', 'AggBids', 'BalState', 'ResPrice', 'ResCap', 'ResAmt', 'PhysFlow_All', 'Import_Sum']
    Extra    : 83 additional columns not in reference (may be valid)

  Missing-value summary (top 15, >0%):
    NTC_Week_GB                                         94.4%  ██████████████████
    NTC_Week_DE_LU                                      94.4%  ██████████████████
    NTC_Week_NL                                         94.4%  ██████████████████
    NTC_Week_FR                          

,Time_UTC,DA_Price,Forecasted_Load,Load_Actual,NetPos,GenFc,WS_Fc_Solar,WS_Fc_Wind_Offshore,WS_Fc_Wind_Onshore,WS_ID_Fc_Solar,...,PhysFlow_All_NL,PhysFlow_All_FR,PhysFlow_All_GB,PhysFlow_All_DE_LU,PhysFlow_All_sum,Import_Sum_NL,Import_Sum_FR,Import_Sum_GB,Import_Sum_DE_LU,Import_Sum_sum
0,2022-01-01 00:00:00+00:00,67.070000,8399.250000,7842.549805,1020.500000,2737.760010,0.0,1750.875000,541.200012,0.0,...,0.0,2361.030029,0.0,0.000000,2361.030029,727.585022,0.0,100.959999,340.609985,1169.155029
1,2022-01-01 00:15:00+00:00,67.070000,8324.169922,7711.500000,NaN,NaN,0.0,1750.875000,541.200012,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,101.540001,282.649994,384.190002
2,2022-01-01 00:30:00+00:00,67.070000,8218.240234,7627.310059,NaN,NaN,0.0,1750.875000,541.200012,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,101.550003,328.089996,429.640015
3,2022-01-01 00:45:00+00:00,67.070000,8113.310059,7453.069824,NaN,NaN,0.0,1750.875000,541.200012,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,100.779999,256.619995,357.399994
4,2022-01-01 01:00:00+00:00,75.110001,8018.529785,7300.430176,873.500000,2349.790039,0.0,1539.352539,493.500000,0.0,...,0.0,2603.985107,0.0,65.379997,2669.364990,1070.037476,0.0,5.060000,144.720001,1219.817505
5,2022-01-01 01:15:00+00:00,75.110001,7931.279785,7199.259766,NaN,NaN,0.0,1539.352539,493.500000,0.0,...,NaN,NaN,0.0,211.039993,211.039993,NaN,NaN,4.280000,148.089996,152.369995
6,2022-01-01 01:30:00+00:00,75.110001,7872.350098,7162.669922,NaN,NaN,0.0,1539.352539,493.500000,0.0,...,NaN,NaN,0.0,381.970001,381.970001,NaN,NaN,4.300000,156.970001,161.270004
7,2022-01-01 01:45:00+00:00,75.110001,7784.770020,7083.560059,NaN,NaN,0.0,1539.352539,493.500000,0.0,...,NaN,NaN,0.0,322.250000,322.250000,NaN,NaN,31.790001,143.479996,175.270004
8,2022-01-01 02:00:00+00:00,50.910000,7699.000000,7016.430176,730.400024,2784.682617,0.0,1330.614990,442.850006,0.0,...,0.0,2893.215088,0.0,0.000000,2893.215088,344.429993,0.0,551.479980,243.179993,1139.089966
9,2022-01-01 02:15:00+00:00,50.910000,7608.209961,7026.580078,NaN,NaN,0.0,1330.614990,442.850006,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,576.940002,217.610001,794.549988


## Cell 6 — Full Historical Download

> **Gate**: set `PREVIEW_ONLY = False` in Cell 2 once the Data Health Report looks correct.

Batches are appended to the `.csv.gz` immediately after each fetch (crash-safe).  
Set `RESUME_FROM_BATCH = N` to skip the first N batches after a crash.

In [6]:
RESUME_FROM_BATCH: int = 0  # set to N to skip the first N batches

if PREVIEW_ONLY:
    print('Skipped (PREVIEW_ONLY=True). Set PREVIEW_ONLY=False in Cell 2 to proceed.')
else:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    # On a fresh run, remove any stale partial file.
    header_written: bool = False
    if RESUME_FROM_BATCH > 0 and OUTPUT_PATH.exists():
        header_written = True
        logger.info(f'Resuming from batch {RESUME_FROM_BATCH} — appending to existing file.')
    elif OUTPUT_PATH.exists():
        logger.warning(f'Removing stale output file: {OUTPUT_PATH}')
        OUTPUT_PATH.unlink()

    fetch_client = EntsoeDataClient()
    failed_batches: list[tuple[str, str]] = []
    active_batches: list[tuple[pd.Timestamp, pd.Timestamp]] = batches[RESUME_FROM_BATCH:]

    for batch_start, batch_end in tqdm(active_batches, desc='Historical Fetch', unit='batch'):
        label: str = batch_label(batch_start)
        print(f'\n{"=" * 65}')
        print(f'Fetching {label}  ({batch_start.date()} -> {batch_end.date()}) ...')
        t0: float = time.time()

        batch_df: pd.DataFrame | None = None
        for attempt in range(1, MAX_BATCH_RETRIES + 1):
            try:
                # --- 1. Fetch from ENTSO-E ----------------------------------------
                raw_df: pd.DataFrame = fetch_client.fetch_comprehensive_market_data(
                    batch_start, batch_end
                )
                if raw_df is None or raw_df.empty:
                    logger.warning(f'[{label}] Empty API response — skipping batch.')
                    failed_batches.append((label, 'Empty response'))
                    break

                # --- 2. Clean, frequency-check, cast to float32 -------------------
                batch_df = clean_batch(raw_df, label)
                break  # success — exit retry loop

            except Exception as exc:
                if is_rate_limit_error(exc):
                    logger.warning(
                        f'[{label}] 429 Rate Limit on attempt {attempt}/{MAX_BATCH_RETRIES}. '
                        f'Sleeping {RATE_LIMIT_SLEEP_S}s before retry...'
                    )
                    time.sleep(RATE_LIMIT_SLEEP_S)
                    if attempt == MAX_BATCH_RETRIES:
                        logger.error(f'[{label}] Max retries reached after 429. Logging as failed.')
                        failed_batches.append((label, f'429 after {MAX_BATCH_RETRIES} retries'))
                else:
                    logger.error(f'[{label}] FAILED (attempt {attempt}): {exc}')
                    if attempt < 2:
                        logger.warning(f'[{label}] Retrying once after non-rate-limit error...')
                        time.sleep(BATCH_SLEEP_S)
                    else:
                        failed_batches.append((label, str(exc)))
                        break  # give up after 1 retry

        if batch_df is None or batch_df.empty:
            time.sleep(BATCH_SLEEP_S)
            continue

        # --- 3. Log missing-value summary -----------------------------------------
        missing_pct: pd.Series = batch_df.isnull().mean() * 100
        high_missing: pd.Series = missing_pct[missing_pct > 20]
        if not high_missing.empty:
            logger.warning(
                f'[{label}] {len(high_missing)} columns with >20% missing:\n'
                + '\n'.join(f'  {c}: {v:.1f}%' for c, v in high_missing.items())
            )
        else:
            logger.info(f'[{label}] Missing-value check passed (all columns <=20% NaN).')

        # --- 4. Serialise index as ISO-8601 UTC string ----------------------------
        batch_df.index = batch_df.index.strftime('%Y-%m-%dT%H:%M:%SZ')

        # --- 5. Append to .csv.gz (crash-safe) ------------------------------------
        batch_df.to_csv(
            OUTPUT_PATH,
            mode='a',
            header=not header_written,
            compression='gzip',
        )
        header_written = True

        elapsed: float = time.time() - t0
        print(
            f'[{label}] OK  {len(batch_df):>5,} rows  |  '
            f'{len(batch_df.columns)} cols  |  {elapsed:.0f}s'
        )

        # Rate-limit buffer between batches.
        time.sleep(BATCH_SLEEP_S)

    print(f'\n{"=" * 65}')
    print(f'Download complete. Output: {OUTPUT_PATH}')
    if failed_batches:
        print(f'\nFailed batches ({len(failed_batches)}):')
        for lbl, reason in failed_batches:
            print(f'  {lbl}: {reason}')
    else:
        print('All batches succeeded.')

21:57:50 [WARNING] historical_fetch - Removing stale output file: /Users/bosun/Developer/Quant/energy-data/data/historical_be_2022_2026.csv.gz
Historical Fetch:   0%|          | 0/18 [00:00<?, ?batch/s]21:57:50 [INFO] root - Executing comprehensive fetch for BE



Fetching 2022-Q1  (2022-01-01 -> 2022-04-01) ...


21:58:00 [WARNING] shared_logic.entsoe_client - Structural corruption in response: Columns with duplicate values are not supported in stack
21:59:55 [WARNING] shared_logic.entsoe_client - Structural corruption in response: Columns with duplicate values are not supported in stack
22:02:50 [INFO] historical_fetch - [2022-Q1] Frequency check passed: continuous 15min grid confirmed.
22:02:50 [INFO] historical_fetch - [2022-Q1] Forward-filled 27 step-function columns.
22:02:50 [WARNING] historical_fetch - [2022-Q1] 74 columns with >20% missing:
  NetPos: 75.0%
  GenFc: 75.0%
  Gen_Biomass_Actual_Aggregated: 75.0%
  Gen_Fossil_Gas_Actual_Aggregated: 75.0%
  Gen_Fossil_Oil_Actual_Aggregated: 75.0%
  Gen_Hydro_Pumped_Storage_Actual_Aggregated: 87.0%
  Gen_Hydro_Pumped_Storage_Actual_Consumption: 83.3%
  Gen_Hydro_Run_of_river_and_poundage_Actual_Aggregated: 75.0%
  Gen_Nuclear_Actual_Aggregated: 75.0%
  Gen_Other_Actual_Aggregated: 75.0%
  Gen_Solar_Actual_Aggregated: 75.0%
  Gen_Waste_Actual_

[2022-Q1] OK  8,640 rows  |  109 cols  |  300s


Historical Fetch:   6%|▌         | 1/18 [05:59<1:41:59, 359.95s/batch]22:03:50 [INFO] root - Executing comprehensive fetch for BE


...


Fetching 2026-Q2  (2026-04-01 -> 2026-04-10) ...


23:13:37 [WARNING] shared_logic.entsoe_client - Structural corruption in response: Columns with duplicate values are not supported in stack
23:13:47 [WARNING] shared_logic.entsoe_client - Structural corruption in response: Columns with duplicate values are not supported in stack
23:13:54 [INFO] historical_fetch - [2026-Q2] Frequency check passed: continuous 15min grid confirmed.
23:13:54 [INFO] historical_fetch - [2026-Q2] Forward-filled 23 step-function columns.
23:13:54 [WARNING] historical_fetch - [2026-Q2] 65 columns with >20% missing:
  GenFc: 75.0%
  Gen_Biomass_Actual_Aggregated: 75.5%
  Gen_Energy_storage_Actual_Aggregated: 75.5%
  Gen_Fossil_Gas_Actual_Aggregated: 75.5%
  Gen_Fossil_Oil_Actual_Aggregated: 75.5%
  Gen_Hydro_Pumped_Storage_Actual_Aggregated: 88.9%
  Gen_Hydro_Pumped_Storage_Actual_Consumption: 84.5%
  Gen_Hydro_Run_of_river_and_poundage_Actual_Aggregated: 75.5%
  Gen_Nuclear_Actual_Aggregated: 75.5%
  Gen_Other_Actual_Aggregated: 75.5%
  Gen_Solar_Actual_Aggrega

[2026-Q2] OK    864 rows  |  101 cols  |  18s


Historical Fetch: 100%|██████████| 18/18 [1:17:03<00:00, 256.87s/batch]


Download complete. Output: /Users/bosun/Developer/Quant/energy-data/data/historical_be_2022_2026.csv.gz
All batches succeeded.


## Cell 7 — Post-processing

Deduplicates, sorts by UTC timestamp, runs a final `asfreq('15min')` pass, ensures
`Time_UTC` is the first column, and overwrites the `.csv.gz`.

In [7]:
if PREVIEW_ONLY:
    print('Skipped (PREVIEW_ONLY=True).')
elif not OUTPUT_PATH.exists():
    print(f'Output file not found: {OUTPUT_PATH}')
else:
    print(f'Loading {OUTPUT_PATH} ...')
    df_final: pd.DataFrame = pd.read_csv(
        OUTPUT_PATH, index_col=0, compression='gzip'
    )

    # Restore UTC DatetimeIndex.
    df_final.index = pd.to_datetime(df_final.index, utc=True)
    df_final.index.name = 'Time_UTC'

    rows_before: int = len(df_final)

    # Deduplicate and sort.
    df_final = (
        df_final
        .loc[~df_final.index.duplicated(keep='first')]
        .sort_index()
    )
    rows_after: int = len(df_final)
    print(
        f'Dedup: {rows_before:,} -> {rows_after:,} rows '
        f'(removed {rows_before - rows_after:,} duplicates)'
    )

    # Final frequency enforcement: asfreq('15min') fills any remaining gaps
    # and confirms a perfectly continuous 15-min grid for Kaggle consumers.
    df_final = check_and_fill_frequency_gaps(df_final, label='final')

    # Ensure float32 dtype is retained after round-tripping through CSV.
    df_final = cast_numeric_to_float32(df_final)

    # Serialise with Time_UTC as first column (Kaggle-friendly layout).
    df_save = ensure_time_utc_first(df_final.copy())
    df_save.to_csv(OUTPUT_PATH, index=False, compression='gzip')

    size_mb: float = OUTPUT_PATH.stat().st_size / 1_048_576
    print(f'Rows    : {len(df_final):,}')
    print(f'Columns : {len(df_final.columns)}')
    print(f'Saved   : {OUTPUT_PATH}  ({size_mb:.1f} MB compressed)')

Loading /Users/bosun/Developer/Quant/energy-data/data/historical_be_2022_2026.csv.gz ...


23:55:44 [INFO] historical_fetch - [final] Frequency check passed: continuous 15min grid confirmed.
23:55:44 [INFO] historical_fetch - [final] Forward-filled 27 step-function columns.


Dedup: 149,760 -> 149,760 rows (removed 0 duplicates)
Rows    : 149,760
Columns : 109
Saved   : /Users/bosun/Developer/Quant/energy-data/data/historical_be_2022_2026.csv.gz  (13.4 MB compressed)


## Cell 8 — Final Summary Report & dataset_metadata.json

Runs the Data Health Report and writes `data/dataset_metadata.json` for Kaggle upload validation.

In [8]:
df_report: pd.DataFrame = df_preview if PREVIEW_ONLY else df_final
report_label: str = 'preview (3 days)' if PREVIEW_ONLY else 'full dataset'

# Restore DatetimeIndex for reporting (index was serialised to strings for CSV).
if not isinstance(df_report.index, pd.DatetimeIndex):
    df_report = df_report.copy()
    df_report.index = pd.to_datetime(df_report.index, utc=True)
    df_report.index.name = 'Time_UTC'

metadata: dict = run_data_health_report(
    df_report,
    REFERENCE_COLUMNS,
    label=report_label,
)

# ---- Gold-standard validation gates ----------------------------------------
EXPECTED_MIN_ROWS   = 140_000   # ~4 years @ 15-min intervals
EXPECTED_MIN_COLS   = 40        # minimum realistic column count

gates: dict = {
    'row_count_ok'  : metadata['rows'] >= EXPECTED_MIN_ROWS if not PREVIEW_ONLY else True,
    'col_count_ok'  : metadata['columns'] >= EXPECTED_MIN_COLS,
    'freq_ok'       : metadata.get('median_step_min', 0) == 15.0,
    'tz_ok'         : metadata.get('index_tz', '') == 'UTC',
    'no_nan_spikes' : len(metadata.get('nan_spike_warning', {})) == 0,
}
metadata['validation_gates'] = gates

print(f'\n{"=" * 65}')
print('GOLD-STANDARD VALIDATION GATES')
for gate, passed in gates.items():
    icon = 'PASS' if passed else 'FAIL'
    print(f'  [{icon}]  {gate}')

all_passed: bool = all(gates.values())
print(f'\n  Overall : {"ALL GATES PASSED — ready for Kaggle upload" if all_passed else "SOME GATES FAILED — review before upload"}')
print('=' * 65)

# ---- Write dataset_metadata.json -------------------------------------------
if not PREVIEW_ONLY:
    METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(METADATA_PATH, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    print(f'\nMetadata written to: {METADATA_PATH}')
else:
    print('\n(dataset_metadata.json not written in PREVIEW_ONLY mode)')

# ---- head(10) display -------------------------------------------------------
df_display = ensure_time_utc_first(df_report.copy())
print(f'\nColumn order (first 5): {list(df_display.columns[:5])}')
print('\n--- head(10) ---')
try:
    display(df_display.head(10))
except NameError:
    print(df_display.head(10).to_string())


DATA HEALTH REPORT  [full dataset]
  Rows            :  149,760
  Columns         :      109
  Median step     :     15.0 min  [OK]
  Index TZ        :      UTC  [expected: UTC]
  Index range     : 2022-01-01 00:00:00+00:00  ->  2026-04-09 23:45:00+00:00
  float32 cols    :      109 / 109 numeric

  Reference column match:
    Matched  :  25 / 39
    MISSING  : ['Load_Forecast', 'Imb_imbalance_short', 'Imb_imbalance_long', 'Gen_Solar', 'Gen_Wind_Offshore', 'Gen_Wind_Onshore', 'Gen_Nuclear', 'AggBids', 'BalState', 'ResPrice', 'ResCap', 'ResAmt', 'PhysFlow_All', 'Import_Sum']
    Extra    : 84 additional columns not in reference (may be valid)

  Missing-value summary (top 15, >0%):
    Gen_Wind_Offshore_Actual_Consumption                89.5%  █████████████████
    Import_Sum_sum                                      88.4%  █████████████████
    Export_FR                                           86.8%  █████████████████
    Export_NL                                           86.8%  ███

,Time_UTC,DA_Price,Forecasted_Load,Load_Actual,NetPos,GenFc,WS_Fc_Solar,WS_Fc_Wind_Offshore,WS_Fc_Wind_Onshore,WS_ID_Fc_Solar,...,PhysFlow_All_NL,PhysFlow_All_FR,PhysFlow_All_GB,PhysFlow_All_DE_LU,PhysFlow_All_sum,Import_Sum_NL,Import_Sum_FR,Import_Sum_GB,Import_Sum_DE_LU,Import_Sum_sum
0,2022-01-01 00:00:00+00:00,67.070000,8399.250000,7842.549805,1020.500000,2737.760010,0.0,1750.875000,541.200012,0.0,...,0.0,2361.030029,0.0,0.000000,2361.030029,727.585022,0.0,100.959999,340.609985,1169.155029
1,2022-01-01 00:15:00+00:00,67.070000,8324.169922,7711.500000,NaN,NaN,0.0,1750.875000,541.200012,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,101.540001,282.649994,384.190002
2,2022-01-01 00:30:00+00:00,67.070000,8218.240234,7627.310059,NaN,NaN,0.0,1750.875000,541.200012,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,101.550003,328.089996,429.640015
3,2022-01-01 00:45:00+00:00,67.070000,8113.310059,7453.069824,NaN,NaN,0.0,1750.875000,541.200012,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,100.779999,256.619995,357.399994
4,2022-01-01 01:00:00+00:00,75.110001,8018.529785,7300.430176,873.500000,2349.790039,0.0,1539.352539,493.500000,0.0,...,0.0,2603.985107,0.0,65.379997,2669.364990,1070.037476,0.0,5.060000,144.720001,1219.817505
5,2022-01-01 01:15:00+00:00,75.110001,7931.279785,7199.259766,NaN,NaN,0.0,1539.352539,493.500000,0.0,...,NaN,NaN,0.0,211.039993,211.039993,NaN,NaN,4.280000,148.089996,152.369995
6,2022-01-01 01:30:00+00:00,75.110001,7872.350098,7162.669922,NaN,NaN,0.0,1539.352539,493.500000,0.0,...,NaN,NaN,0.0,381.970001,381.970001,NaN,NaN,4.300000,156.970001,161.270004
7,2022-01-01 01:45:00+00:00,75.110001,7784.770020,7083.560059,NaN,NaN,0.0,1539.352539,493.500000,0.0,...,NaN,NaN,0.0,322.250000,322.250000,NaN,NaN,31.790001,143.479996,175.270004
8,2022-01-01 02:00:00+00:00,50.910000,7699.000000,7016.430176,730.400024,2784.682617,0.0,1330.614990,442.850006,0.0,...,0.0,2893.215088,0.0,0.000000,2893.215088,344.429993,0.0,551.479980,243.179993,1139.089966
9,2022-01-01 02:15:00+00:00,50.910000,7608.209961,7026.580078,NaN,NaN,0.0,1330.614990,442.850006,0.0,...,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,576.940002,217.610001,794.549988
